In [ ]:
import os
import torch
import torchvision
import xml.etree.ElementTree as ET
import pandas as pd
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import functional as F
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision import transforms
import torch.optim.lr_scheduler as lr_scheduler

# === CONFIGURATION ===
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_CLASSES = 6
CONF_THRESHOLD = 0.5

TRAIN_IMG_ROOT = "./train_images"
TRAIN_ANN_ROOT = "./train_annotations"
VAL_IMG_ROOT = "./test"
SUBMISSION_PATH = "./submission4.csv"

# === LABELS ===
LABEL_MAP = {
    'crazing': 0,
    'inclusion': 1,
    'patches': 2,
    'pitted_surface': 3,
    'rolled-in_scale': 4,
    'scratches': 5
}
REV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

# === DATA AUGMENTATION ===
def get_transform(train=True):
    transforms_list = []
    if train:
        transforms_list.append(transforms.RandomHorizontalFlip(0.5))
        transforms_list.append(transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2))
        transforms_list.append(transforms.RandomRotation(10))
    transforms_list.append(transforms.ToTensor())
    return transforms.Compose(transforms_list)

class NEUDETDataset(Dataset):
    def __init__(self, img_dir, ann_dir, transforms=None):
        self.img_dir = img_dir
        self.ann_dir = ann_dir
        self.transforms = transforms
        self.imgs = sorted([f for f in os.listdir(img_dir) if f.endswith('.jpg')])

    def __getitem__(self, idx):
        img_name = self.imgs[idx]
        img_path = os.path.join(self.img_dir, img_name)
        ann_path = os.path.join(self.ann_dir, img_name.replace('.jpg', '.xml'))
        image = Image.open(img_path).convert("RGB")

        boxes = []
        labels = []

        tree = ET.parse(ann_path)
        root = tree.getroot()
        for obj in root.findall('object'):
            label = obj.find('name').text
            bbox = obj.find('bndbox')
            xmin = int(float(bbox.find('xmin').text))
            ymin = int(float(bbox.find('ymin').text))
            xmax = int(float(bbox.find('xmax').text))
            ymax = int(float(bbox.find('ymax').text))
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(LABEL_MAP[label])

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        target = {'boxes': boxes, 'labels': labels, 'image_id': torch.tensor([idx])}

        if self.transforms:
            image = self.transforms(image)
        else:
            image = F.to_tensor(image)

        return image, target

    def __len__(self):
        return len(self.imgs)

def get_model():
    model = fasterrcnn_resnet50_fpn(pretrained=True)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
    return model.to(DEVICE)

def collate_fn(batch):
    return tuple(zip(*batch))

def train_model(model, train_loader, epochs=10, save_path="best_model.pth"):
    model.train()
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=1e-4)
    scheduler = lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)  # Learning Rate Scheduling

    best_loss = float('inf')

    for epoch in range(epochs):
        total_loss = 0
        for images, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            loss = sum(loss for loss in loss_dict.values())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1} Average Loss: {avg_loss:.4f}")

        # Save model if current epoch has lower loss
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), save_path)
            print(f"[INFO] Saved new best model with loss {best_loss:.4f} at epoch {epoch+1}")

        scheduler.step()

def predict_and_export(model, val_dir, save_csv_path, save_img_dir="output"):
    os.makedirs(save_img_dir, exist_ok=True)
    model.eval()
    rows = []

    val_images = sorted([f for f in os.listdir(val_dir) if f.lower().endswith('.jpg')])

    for img_file in tqdm(val_images, desc="Predicting"):
        img_path = os.path.join(val_dir, img_file)
        image = Image.open(img_path).convert("RGB")
        image_tensor = F.to_tensor(image).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            output = model(image_tensor)[0]

        boxes = output["boxes"].cpu().numpy()
        scores = output["scores"].cpu().numpy()
        labels = output["labels"].cpu().numpy()

        cf_list, xmin_list, ymin_list, xmax_list, ymax_list = [], [], [], [], []

        # Plot image
        fig, ax = plt.subplots(1)
        ax.imshow(image)

        for box, score, label in zip(boxes, scores, labels):
            xmin, ymin, xmax, ymax = map(int, box.tolist())
            cf_list.append(f"{score:.2f}")
            xmin_list.append(str(xmin))
            ymin_list.append(str(ymin))
            xmax_list.append(str(xmax))
            ymax_list.append(str(ymax))

            # Draw box
            rect = patches.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                     linewidth=2, edgecolor='red', facecolor='none')
            ax.add_patch(rect)
            ax.text(xmin, ymin - 5, f"{REV_LABEL_MAP[label.item()]} {score:.2f}",
                    color='red', fontsize=8, weight='bold')

        # Save image
        fig.savefig(os.path.join(save_img_dir, img_file), bbox_inches='tight', dpi=150)
        plt.close(fig)

        rows.append({
            "ID": img_file,
            "label": REV_LABEL_MAP[labels[0].item()] if len(labels) > 0 else "none",
            "cf": " ".join(cf_list),
            "xmin": " ".join(xmin_list),
            "ymin": " ".join(ymin_list),
            "xmax": " ".join(xmax_list),
            "ymax": " ".join(ymax_list),
        })

    df = pd.DataFrame(rows)
    df.fillna("none", inplace=True)
    df.to_csv(save_csv_path, index=False)
    print(f"[INFO] Saved CSV to {save_csv_path}")
    print(f"[INFO] Saved annotated images to {save_img_dir}/")

if __name__ == "__main__":
    # Initialize dataset and dataloader
    train_dataset = NEUDETDataset(TRAIN_IMG_ROOT, TRAIN_ANN_ROOT)
    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)

    # Get model
    model = get_model()

    # Load checkpoint if exists
    if os.path.exists("best_model.pth"):
        model.load_state_dict(torch.load("best_model.pth", map_location=DEVICE))
        print("[INFO] Loaded existing best_model.pth for continued training.")
    else:
        print("[INFO] No existing checkpoint found. Starting training from scratch.")

    # Train model
    train_model(model, train_loader, epochs=5, save_path="best_model_finetune.pth")

    # Load best model for prediction
    model.load_state_dict(torch.load("best_model_finetune.pth", map_location=DEVICE))
    print("[INFO] Loaded best model for prediction.")

    # Predict and export results
    predict_and_export(model, VAL_IMG_ROOT, SUBMISSION_PATH, save_img_dir="output")



[INFO] Loaded existing best_model.pth for continued training.


Epoch 1: 100%|██████████| 360/360 [2:48:24<00:00, 28.07s/it]   


Epoch 1 Average Loss: 0.2097
[INFO] Saved new best model with loss 0.2097 at epoch 1


Epoch 2: 100%|██████████| 360/360 [3:05:00<00:00, 30.83s/it]    


Epoch 2 Average Loss: 0.2006
[INFO] Saved new best model with loss 0.2006 at epoch 2


Epoch 3:  25%|██▌       | 90/360 [40:57<3:31:04, 46.91s/it]